<a href="https://colab.research.google.com/github/JJualo/Arabic-Token-Tax/blob/main/ArabicTextTax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q openai tiktoken
import pandas as pd, math, time
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("GPT_key"))

In [12]:
!git clone -q https://github.com/SinaLab/ArBanking77.git
!ls ArBanking77/data

Banking77_Arabized_Moroccan_test.csv  Banking77_Arabized_Saudi_test.csv
Banking77_Arabized_MSA_PAL_train.csv  Banking77_Arabized_Tunisian_test.csv
Banking77_Arabized_MSA_PAL_val.csv    Banking77_full_corpus.csv
Banking77_Arabized_MSA_test.csv       Banking77_intents.csv
Banking77_Arabized_PAL_test.csv


In [13]:
import pandas as pd

df = pd.read_csv("ArBanking77/data/Banking77_Arabized_Saudi_test.csv")
print(df.shape)
print(df.columns.tolist())
df.head(5)

(3580, 2)
['label', 'text']


,label,text
0,وصول البطاقة,كيف أقدر أعرف مكان بطاقتي؟
1,وصول البطاقة,للحين ما استلمت بطاقتي الجديدة، طالبها من أكثر...
2,وصول البطاقة,طلبت بطاقة بس ما وصلت الله يخليك ساعدني!
3,وصول البطاقة,فيه طريقة اعرف متى توصل بطاقتي؟
4,وصول البطاقة,ما حصلت على بطاقتي للحين


In [14]:
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print(len(df))

3542


In [15]:
df['label'].unique()

array(['وصول البطاقة', 'ربط البطاقة', 'سعر الصرف',
       'سعر الصرف الخاطئ للدفع بالبطاقة', 'رسوم إضافية على كشف الحساب',
       'في انتظار السحب النقدي', 'دعم العملات الورقية',
       'تقدير تسليم البطاقة', 'تعبئة تلقائية', 'البطاقة لا تعمل',
       'الصرف عبر التطبيق', 'البطاقة المفقودة أو المسروقة', 'حد السن',
       'رمز التعريف الشخصي محظور', 'عدم التلامس لا يعمل',
       'إعادة الشحن عن طريق رسوم التحويل المصرفي', 'في انتظار التعبئة',
       'إلغاء التحويل', 'حدود التعبئة', 'تلقي مبلغ خاطئ من النقد',
       'تم تحصيل رسوم الدفع بالبطاقة', 'لم يستلم المستلم التحويل',
       'البطاقات والعملات المدعومة', 'الحصول على بطاقة افتراضية',
       'قبول البطاقة', 'عادت تعبئة الرصيد',
       'لم يتم تحديث الرصيد بعد الشيك أو الإيداع النقدي',
       'الدفع بالبطاقة غير معترف به', 'تحرير التفاصيل الشخصية',
       'لماذا التحقق من الهوية', 'غير قادر على التحقق من الهوية',
       'الحصول على بطاقة فعلية', 'فيزا أو ماستر كارد',
       'التعبئة عن طريق البطاقة', 'حدود البطاقة التي يمكن التخلص من

In [16]:
GROUPS = {
    "cards": ["وصول البطاقة", "تقدير تسليم البطاقة", "طلب بطاقة فعلية", "الحصول على بطاقة فعلية",
              "الحصول على بطاقة احتياطية", "بطاقة على وشك الانتهاء", "تفعيل بطاقتي", "ربط البطاقة",
              "البطاقة لا تعمل", "عدم التلامس لا يعمل", "قبول البطاقة", "الحصول على بطاقة افتراضية",
              "الحصول على بطاقة افتراضية يمكن التخلص منها", "حدود البطاقة التي يمكن التخلص منها",
              "البطاقة الافتراضية لا تعمل", "تغيير رمز التعريف الشخصي", "رمز التعريف الشخصي محظور",
              "آبل باي أو جوجل باي", "فيزا أو ماستر كارد"],
    "security": ["البطاقة المفقودة أو المسروقة", "الهاتف المفقود أو المسروق", "بطاقة مخترقة",
                 "الدفع بالبطاقة غير معترف به", "السحب النقدي غير معترف به", "الدفع المباشر للدين غير معروف"],
    "transfers": ["نقل إلى الحساب", "فشل التحويل", "رفض التحويل", "انتظار التحويل", "إلغاء التحويل",
                  "توقيت التحويل", "لم يستلم المستلم التحويل", "المستفيد غير مسموح به", "تلقي الأموال",
                  "لم يتم تحديث الرصيد بعد التحويل المصرفي"],
    "top_ups": ["في انتظار التعبئة", "تعبئة تلقائية", "تعبئة الرصيد نقدًا أو بشيك", "حدود التعبئة",
                "التعبئة عن طريق البطاقة", "فشل التعبئة", "تحقق من تعبئة الرصيد", "عادت تعبئة الرصيد",
                "لم يتم تحديث الرصيد بعد الشيك أو الإيداع النقدي"],
    "cash_atm": ["في انتظار السحب النقدي", "تلقي مبلغ خاطئ من النقد", "رفض السحب النقدي",
                 "دعم أجهزة الصراف الآلي", "ابتلاع البطاقة"],
    "fees_exchange": ["تم تحصيل رسوم الدفع بالبطاقة", "سعر الصرف الخاطئ للدفع بالبطاقة",
                      "إعادة الشحن عن طريق رسوم التحويل المصرفي", "التعبئة عن طريق شحن البطاقة",
                      "رسوم السحب النقدي", "سعر صرف خاطئ للسحب النقدي", "تحصيل رسوم التحويل",
                      "رسوم إضافية على كشف الحساب", "سعر الصرف", "الصرف عبر التطبيق", "رسوم الصرف",
                      "دعم العملات الورقية", "البطاقات والعملات المدعومة"],
    "card_payments": ["الدفع بالبطاقة المعلق", "تم رفض الدفع بالبطاقة", "تم التحصيل مرتين",
                      "إرجاع الدفع بالبطاقة؟", "طلب استرداد", "استرداد الأموال غير مرئي"],
    "account": ["انهاء حساب", "تحرير التفاصيل الشخصية", "تحقق من هويتي", "لماذا التحقق من الهوية",
                "غير قادر على التحقق من الهوية", "التحقق من مصدر الأموال", "حد السن", "دعم البلد",
                "نسيان رمز المرور"],
}
DEPTS = list(GROUPS)

intent_to_dept = {intent: dept for dept, intents in GROUPS.items() for intent in intents}
df["dept"] = df["label"].map(intent_to_dept)
print("unmapped:", df["dept"].isna().sum())   # 0

unmapped: 0


In [17]:
from sklearn.model_selection import train_test_split

test, dev = train_test_split(
    df,
    test_size=500,
    stratify=df["dept"],
    random_state=42
)

print("dev:", len(dev), "| test:", len(test))

dev: 500 | test: 3042


In [18]:
ROUTER_PROMPT = """You route Saudi bank customer messages to the right department.
Departments:
- cards: ordering, delivery, activation, PIN; the card, contactless, virtual card or Apple Pay not working; where the card is accepted
- security: lost or stolen card or phone, unrecognised payments, compromised card
- transfers: sending or receiving money, including receiving money in other currencies; failed or pending transfers
- top_ups: adding money to the account
- cash_atm: cash withdrawals and ATM problems
- fees_exchange: any fee or exchange rate question
- card_payments: a specific card payment that is pending or charged twice, or a refund
- account: identity verification, personal details, closing the account, which countries the bank serves
Always choose the closest department, never refuse.
Reply with exactly one department name from the list, nothing else."""

REPLY_PROMPT = """You are a friendly customer service assistant for a Saudi bank.

Bank policy:
{policy}

Write a reply in clear, warm, natural Modern Standard Arabic (simple MSA, not overly formal), using at most 2 short sentences.

Follow these rules strictly:
- Identify the customer's exact question or problem before answering.
- Answer only that question or problem. Every sentence must be directly relevant to what the customer asked.
- Never switch to a different issue or suggest an unrelated action. For example, if a customer says their card was declined at a restaurant, do not give password or PIN-changing instructions unless the policy explicitly identifies that as the solution.
- Give the most useful answer or action supported by the policy first.
- Only state facts, steps, fees, eligibility requirements, limits, app navigation instructions, or timelines that are explicitly mentioned in the policy.
- Never guess, assume, or add plausible-sounding details to fill gaps.
- If the policy does not directly address the customer's issue or does not provide a necessary detail, clearly say that the bank's support team can confirm the information or help with the issue instead of guessing.
- Include instructions or next steps only when they are relevant to the customer's exact problem and supported by the policy.
- Never invent information or promise an outcome.
- Avoid greetings, unnecessary apologies, repetition, and generic filler.

Return only the customer-facing reply."""

In [19]:
POLICY = {
    "cards": "Physical cards arrive within 5-7 working days and can be tracked in the app under Cards. Activate a new card in the app before first use. PINs can be changed in the app or at our ATMs; after 3 wrong attempts the PIN is blocked and can be unblocked in the app. Virtual and disposable virtual cards are free and created instantly in the app. Cards work with Apple Pay and Google Pay.",
    "security": "If a card or phone is lost, stolen or compromised, freeze the card immediately in the app or call our 24/7 support line. A replacement card is free. For any payment, cash withdrawal or direct debit the customer does not recognise, freeze the card and file a dispute in the app; it is reviewed within 5 working days. Never share your PIN or one-time codes with anyone, including bank staff.",
    "transfers": "Transfers between our accounts are instant. Local transfers to other Saudi banks arrive within 1 working day; international transfers take 1-3 working days. Pending transfers can be cancelled in the app until processed. Transfers fail if beneficiary details are wrong or the beneficiary is not verified; new beneficiaries are verified by SMS code. Incoming money is accepted in SAR, USD, EUR and GBP and converted to SAR.",
    "top_ups": "Accounts can be topped up by bank transfer, debit card or cash deposit at our ATMs. Card top-ups are instant, bank transfers take 1 working day, cheque deposits 2 working days. The daily top-up limit is 20,000 SAR. Auto top-up can be set in the app. Failed or reverted top-ups return to the original source within 3-5 working days.",
    "cash_atm": "Cash can be withdrawn at any ATM in Saudi Arabia and abroad. If an ATM gives the wrong amount or declines a withdrawal, report it in the app with the ATM location and time; it is reviewed within 5 working days. If an ATM keeps the card, freeze it in the app and order a free replacement.",
    "fees_exchange": "Card payments in SAR are free. Foreign currency payments and withdrawals use the Visa/Mastercard rate plus a 2.5% fee, shown in the app before confirming. Local ATM withdrawals are free; withdrawals abroad cost 15 SAR each. Local transfers are free; international transfers cost 35 SAR. Topping up by card is free. Supported currencies: SAR, USD, EUR, GBP and AED.",
    "card_payments": "Card payments can show as pending for up to 3 working days. Declined payments are usually caused by low balance, a frozen card or merchant restrictions. If a payment was charged twice, the duplicate usually disappears within 3 working days; if not, file a dispute in the app. Merchant refunds take 5-10 working days to appear.",
    "account": "Customers verify their identity with their national ID or Iqama through Nafath in the app, as required by Saudi regulations. Customers must be 18 or older. Phone number and address can be updated in the app. To close an account, withdraw the balance and request closure in the app; it is processed within 5 working days. Forgotten passwords can be reset from the login screen.",
}

In [20]:
print(len(POLICY))
print(POLICY["transfers"])

8
Transfers between our accounts are instant. Local transfers to other Saudi banks arrive within 1 working day; international transfers take 1-3 working days. Pending transfers can be cancelled in the app until processed. Transfers fail if beneficiary details are wrong or the beneficiary is not verified; new beneficiaries are verified by SMS code. Incoming money is accepted in SAR, USD, EUR and GBP and converted to SAR.


In [21]:
import os
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def with_retries(func, *args):
    last = ""
    for attempt in range(6):
        try:
            return func(*args)
        except Exception as e:
            last = str(e)
            if "insufficient_quota" in last or "no credits" in last:
                break
            time.sleep(5 * (attempt + 1))         # wait 5, 10, 15... seconds
    return {"error": last[:200]}

def parallel(func, items, workers=5):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(tqdm(pool.map(func, items), total=len(items)))

def load_or_run(name, make):
    path = f"{name}.csv"
    if os.path.exists(path):
        print("loaded saved results:", path)
        return pd.read_csv(path)
    res = make()
    res.to_csv(path, index=False)
    print("saved new results:", path)
    return res

In [22]:
import math
import time

def route(model, message):
    """Predict the department and record confidence, token usage, and time."""
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        temperature=0,
        max_completion_tokens=10,
        logprobs=True,
        messages=[
            {"role": "system", "content": ROUTER_PROMPT},
            {"role": "user", "content": message},
        ],
    )

    return {
        "pred": response.choices[0].message.content.strip(),
        "confidence": math.exp(
            sum(token.logprob for token in response.choices[0].logprobs.content)
        ),
        "route_in": response.usage.prompt_tokens,
        "route_out": response.usage.completion_tokens,
        "route_sec": time.time() - start,
    }


def write_reply(model, message, dept, prompt):
    start = time.time()
    policy = POLICY.get(dept, "No policy available.")

    response = client.chat.completions.create(
        model=model,
        temperature=0,
        max_completion_tokens=120,
        messages=[
            {"role": "system", "content": prompt.format(policy=policy)},
            {"role": "user", "content": message},
        ],
    )

    return {
        "reply": response.choices[0].message.content.strip(),
        "reply_in": response.usage.prompt_tokens,
        "reply_out": response.usage.completion_tokens,
        "reply_sec": time.time() - start,
    }

In [23]:
row = dev.iloc[4]
print("customer:", row["text"], "| correct:", row["dept"])

r1 = route("gpt-4o", row["text"])
r2 = write_reply("gpt-4o", row["text"], r1["pred"], REPLY_PROMPT)

customer: وش أعلى حد لقوة الشحن؟ | correct: top_ups


In [24]:
def one_model(model, message):
    r1 = route(model, message)
    r2 = write_reply(model, message, r1["pred"], REPLY_PROMPT)
    return r1 | r2

def run_one_model(model, data):
    results = parallel(lambda text: with_retries(one_model, model, text), data["text"].tolist())
    res = data.reset_index(drop=True).rename(columns={"dept": "true_dept"})
    res = pd.concat([res, pd.DataFrame(results)], axis=1)
    res["correct"] = res["pred"] == res["true_dept"]
    return res

Costs here are in US dollars, as billed by OpenAI. The presentation converts them to riyals at 3.75 SAR per USD.

In [25]:
PRICES = {
    "gpt-4o": (2.50, 10.00),
    "gpt-4o-mini": (0.15, 0.60),
}  # Dollars per 1 million input/output tokens


def part_cost(results, model, part):
    """Calculate the cost for either routing or reply generation."""
    input_price, output_price = PRICES[model]

    return (
        results[f"{part}_in"] * input_price
        + results[f"{part}_out"] * output_price
    ) / 1_000_000


def summary(name, results, route_model, reply_model):
    total_cost = (
        part_cost(results, route_model, "route")
        + part_cost(results, reply_model, "reply")
    )

    return {
        "design": name,
        "messages": len(results),
        "accuracy %": round(results["correct"].mean() * 100, 1),
        "invalid": int((~results["pred"].isin(DEPTS)).sum()),
        "cost per 1,000 correct $": round(
            total_cost.sum() / results["correct"].sum() * 1000, 2
        ),
        "avg seconds": round(
            (results["route_sec"] + results["reply_sec"]).mean(), 2
        ),
    }

## **Experiment 1: Baseline (gpt-4o does everything)**

In [26]:
big_dev  = load_or_run("big_dev", lambda: run_one_model("gpt-4o", dev))
big_test = load_or_run("big_test",   lambda: run_one_model("gpt-4o", test))

loaded saved results: big_dev.csv
loaded saved results: big_test.csv


## **Experiment 2: Mini alone**

In [27]:
small_dev  = load_or_run("small_dev",  lambda: run_one_model("gpt-4o-mini", dev))
small_test = load_or_run("small_test", lambda: run_one_model("gpt-4o-mini", test))

pd.DataFrame([summary("baseline: gpt-4o", big_test, "gpt-4o", "gpt-4o"),
              summary("mini alone", small_test, "gpt-4o-mini", "gpt-4o-mini")])

loaded saved results: small_dev.csv
loaded saved results: small_test.csv


,design,messages,accuracy %,invalid,"cost per 1,000 correct $",avg seconds
0,baseline: gpt-4o,3042,82.0,1,2.16,1.66
1,mini alone,3042,79.0,3,0.14,1.76


## **Experiment 3: Confidence cascade (tested, then rejected)**

In [28]:
both = small_dev[["text", "pred", "confidence", "correct"]].copy()
both["valid"] = both["pred"].isin(DEPTS)
both["small_cost"] = part_cost(small_dev, "gpt-4o-mini", "route") + part_cost(small_dev, "gpt-4o-mini", "reply")

big_cost = part_cost(big_dev, "gpt-4o", "route") + part_cost(big_dev, "gpt-4o", "reply")
both["big_correct"] = both["text"].map(big_dev.set_index("text")["correct"])
both["big_cost"] = both["text"].map(big_cost.set_axis(big_dev["text"]))

def cascade(cutoff):
    keep = both["valid"] & (both["confidence"] >= cutoff)     # mini keeps these
    correct = both["correct"].where(keep, both["big_correct"])
    cost = both["small_cost"] + both["big_cost"].where(~keep, 0)
    return {"cutoff": cutoff, "escalated %": round((~keep).mean() * 100, 1),
            "accuracy %": round(correct.mean() * 100, 1),
            "cost per 1,000 correct $": round(cost.sum() / correct.sum() * 1000, 2)}

pd.DataFrame([cascade(c) for c in [0.9, 0.99, 0.999, 0.9999, 0.99999, 0.999999]])

,cutoff,escalated %,accuracy %,"cost per 1,000 correct $"
0,0.900000,5.0,78.8,0.25
1,0.990000,11.6,79.8,0.39
2,0.999000,16.0,79.8,0.49
3,0.999900,24.6,80.8,0.67
4,0.999990,48.0,82.2,1.17
5,0.999999,82.2,83.0,1.90


In [29]:
for name, c in [("right", both[both["correct"]]["confidence"]),
                ("wrong", both[~both["correct"]]["confidence"])]:
    print(f"{name}: {len(c)} | above 0.99: {(c > 0.99).mean():.0%} | above 0.9999: {(c > 0.9999).mean():.0%}")

right: 392 | above 0.99: 94% | above 0.9999: 82%
wrong: 108 | above 0.99: 67% | above 0.9999: 52%


Why it didn’t work: mini is overconfident. Half of its wrong answers had confidence above 0.9999, the same range as its right answers, and matching the baseline’s accuracy meant escalating 83% of messages, which removed the savings.



Conclusion: a confidence cascade can't match gpt-4o's accuracy without escalating almost everything.

## **Experiment 4: Split design (gpt-4o routes, mini writes)**

Each model does the job it’s best at: gpt-4o makes the hard decision (routing), and mini writes the reply. Both designs use the same routing (gpt-4o’s, already saved) and the same REPLY_PROMPT, so the only difference is which model writes the reply.

In [30]:
def rerun_replies(route_res, model, prompt):
    """Keep the saved routing, and only (re)write the replies with the given model and prompt."""
    items = list(zip(route_res["text"], route_res["pred"]))
    replies = parallel(lambda p: with_retries(write_reply, model, p[0], p[1], prompt), items)
    keep = route_res[["text", "true_dept", "pred", "correct", "route_in", "route_out", "route_sec"]]
    return pd.concat([keep.reset_index(drop=True), pd.DataFrame(replies)], axis=1)

baseline_dev     = big_dev     # gpt-4o already routed AND replied with REPLY_PROMPT
split_design_dev = load_or_run("split_design_dev", lambda: rerun_replies(big_dev, "gpt-4o-mini", REPLY_PROMPT))

pd.DataFrame([summary("baseline (dev)",     baseline_dev,     "gpt-4o", "gpt-4o"),
              summary("split design (dev)", split_design_dev, "gpt-4o", "gpt-4o-mini")])

loaded saved results: split_design_dev.csv


,design,messages,accuracy %,invalid,"cost per 1,000 correct $",avg seconds
0,baseline (dev),500,83.2,0,2.13,1.66
1,split design (dev),500,83.2,0,0.72,1.83


Result: kept exactly the same accuracy as the baseline, 82%, because gpt-4o still makes every decision. And it cut the cost per correct answer by 66%. It’s only 0.08 seconds slower per message

Experiment 5: Reply quality (blind judging)

In [31]:
import json
import random

!pip install -q anthropic
import anthropic

aclient = anthropic.Anthropic(api_key=userdata.get("claudie"))
JUDGE_MODEL = "claude-sonnet-5-5"

JUDGE_PROMPT = """You evaluate replies from a Saudi bank's customer service assistant.
You get the customer message, the bank policy, and two replies, A and B.
For each reply, decide if it is OK: consistent with the policy (no invented facts), helpful and polite.
Then say which reply is better overall: "A", "B" or "tie".
Answer only in JSON: {"A_ok": true or false, "B_ok": true or false, "better": "A" or "B" or "tie"}"""

def ask_judge(content):
    r = aclient.messages.create(
        model=JUDGE_MODEL, max_tokens=300,
        thinking={"type": "between_tools"},
        system=JUDGE_PROMPT,
        messages=[{"role": "user", "content": content}])
    text = "".join(b.text for b in r.content if b.type == "text")      # skip thinking, keep the answer
    return json.loads(text[text.find("{"): text.rfind("}") + 1])       # keep only the JSON part

def judge_pair(i, row):
    mini_first = random.Random(i).random() < 0.5          # random but repeatable order
    A, B = (row["mini_reply"], row["big_reply"]) if mini_first else (row["big_reply"], row["mini_reply"])
    content = (f"Customer: {row['text']}\nPolicy: {POLICY.get(row['pred'], 'No policy available.')}\n"
               f"Reply A: {A}\nReply B: {B}")
    v = ask_judge(content)
    names = {"A": "mini" if mini_first else "big", "B": "big" if mini_first else "mini", "TIE": "tie"}
    return {"mini_ok": v["A_ok"] if mini_first else v["B_ok"],
            "big_ok": v["B_ok"] if mini_first else v["A_ok"],
            "better": names[str(v["better"]).strip().upper()]}

def run_judge(pairs, workers=5):
    sample = pairs.sample(100, random_state=42).to_dict("records")
    out = parallel(lambda x: with_retries(judge_pair, x[0], x[1]), list(enumerate(sample)), workers)
    return pd.DataFrame(out)

def judge_summary(name, v):
    ok = v.dropna(subset=["better"])
    return {"round": name, "judged": len(ok),
            "mini OK %": round(ok["mini_ok"].mean() * 100), "gpt-4o OK %": round(ok["big_ok"].mean() * 100),
            "mini better": (ok["better"] == "mini").sum(), "tie": (ok["better"] == "tie").sum(),
            "gpt-4o better": (ok["better"] == "big").sum()}

removed old judge_dev.csv


In [34]:
   !unzip -o -q results.zip judge_dev.csv
   !ls judge_dev.csv

judge_dev.csv


In [35]:
pairs_dev = split_design_dev[["text", "pred"]].copy()
pairs_dev["mini_reply"] = split_design_dev["reply"]
pairs_dev["big_reply"] = pairs_dev["text"].map(baseline_dev.set_index("text")["reply"])

judge_pair(0, pairs_dev.iloc[0].to_dict())

{'mini_ok': True, 'big_ok': True, 'better': 'tie'}

In [36]:
judge_dev = load_or_run("judge_dev", lambda: run_judge(pairs_dev))
pd.DataFrame([judge_summary("dev, Claude judge", judge_dev)])

loaded saved results: judge_dev.csv


,round,judged,mini OK %,gpt-4o OK %,mini better,tie,gpt-4o better
0,"dev, Claude judge",98,83,89,40,19,39


### Final eval: the chosen design on all 3,042 unseen test messages, plus blind reply judging on 100 test pairs.

In [37]:
split_design_test = load_or_run("split_design_test", lambda: rerun_replies(big_test, "gpt-4o-mini", REPLY_PROMPT))

pd.DataFrame([summary("baseline (test)",     big_test,          "gpt-4o", "gpt-4o"),
              summary("split design (test)", split_design_test, "gpt-4o", "gpt-4o-mini")])

loaded saved results: split_design_test.csv


,design,messages,accuracy %,invalid,"cost per 1,000 correct $",avg seconds
0,baseline (test),3042,82.0,1,2.16,1.66
1,split design (test),3042,82.0,1,0.73,1.74


In [38]:
pairs_test = split_design_test[["text", "pred"]].copy()
pairs_test["mini_reply"] = split_design_test["reply"]
pairs_test["big_reply"] = pairs_test["text"].map(big_test.set_index("text")["reply"])

judge_test = load_or_run("judge_test", lambda: run_judge(pairs_test))
pd.DataFrame([judge_summary("test, Claude judge", judge_test)])

loaded saved results: judge_test.csv


,round,judged,mini OK %,gpt-4o OK %,mini better,tie,gpt-4o better
0,"test, Claude judge",99,74,88,36,27,36


### the final design on 300 messages each in MSA, Palestinian and Moroccan Arabic (same 8 departments), compared with the Saudi test result.

In [39]:
!pip install -q tiktoken
import tiktoken
enc = tiktoken.get_encoding("o200k_base")     # the tokenizer gpt-4o uses

DIALECT_FILES = {"MSA": "Banking77_Arabized_MSA_test.csv",
                 "Palestinian": "Banking77_Arabized_PAL_test.csv",
                 "Moroccan": "Banking77_Arabized_Moroccan_test.csv"}

def run_design(data):
    """Our design on any data: gpt-4o routes, mini writes the reply."""
    routes = parallel(lambda t: with_retries(route, "gpt-4o", t), data["text"].tolist())
    res = data.reset_index(drop=True).rename(columns={"dept": "true_dept"})
    res = pd.concat([res, pd.DataFrame(routes)], axis=1)
    res["correct"] = res["pred"] == res["true_dept"]
    return rerun_replies(res, "gpt-4o-mini", REPLY_PROMPT)

def avg_tokens(texts):
    return round(texts.map(lambda t: len(enc.encode(t))).mean(), 1)

rows = [summary("Saudi (test)", split_design_test, "gpt-4o", "gpt-4o-mini") | {"avg message tokens": avg_tokens(test["text"])}]

for name, file in DIALECT_FILES.items():
    d = pd.read_csv(f"ArBanking77/data/{file}").drop_duplicates(subset="text")
    d["dept"] = d["label"].map(intent_to_dept)
    print(name, "| unmapped intents:", d["dept"].isna().sum())
    d = d.dropna(subset=["dept"]).sample(300, random_state=42)
    res = load_or_run(f"dialect_{name}", lambda: run_design(d))
    rows.append(summary(name, res, "gpt-4o", "gpt-4o-mini") | {"avg message tokens": avg_tokens(d["text"])})

pd.DataFrame(rows)

MSA | unmapped intents: 0
loaded saved results: dialect_MSA.csv
Palestinian | unmapped intents: 0
loaded saved results: dialect_Palestinian.csv
Moroccan | unmapped intents: 0
loaded saved results: dialect_Moroccan.csv


,design,messages,accuracy %,invalid,"cost per 1,000 correct $",avg seconds,avg message tokens
0,Saudi (test),3042,82.0,1,0.73,1.74,14.8
1,MSA,300,88.0,0,0.69,1.75,16.2
2,Palestinian,300,83.0,0,0.72,1.74,14.6
3,Moroccan,300,84.0,0,0.72,1.75,17.4


In [ ]:
!zip -q results.zip *.csv
from google.colab import files
files.download("results.zip")

In [64]:
import tiktoken
enc = tiktoken.get_encoding("o200k_base")   # the tokenizer gpt-4o and gpt-4o-mini use

EN = "We woke up early to watch the sunrise from the top of the mountain."
AR = "استيقظنا في وقت مبكر لنشاهد شروق الشمس من قمة الجبل."

def show(name, text):
    ids = enc.encode(text)
    pieces = [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in ids]
    print(f"{name}: {len(ids)} tokens")
    print(" | ".join(pieces))
    print()
    return len(ids)

n_en = show("English", EN)
n_ar = show("Arabic", AR)

print(f"Arabic needs {(n_ar - n_en) / n_en:.0%} more tokens for the same sentence.")

English: 15 tokens
We |  woke |  up |  early |  to |  watch |  the |  sunrise |  from |  the |  top |  of |  the |  mountain | .

Arabic: 20 tokens
است | يق | ظ | نا |  في |  وقت |  مب | كر |  لن | ش | اهد |  ش | روق |  الشمس |  من |  ق | مة |  الج | بل | .

Arabic needs 33% more tokens for the same sentence.


In [57]:
!pip install -q "gradio>=5,<6"
import gradio as gr, html, threading, time, tiktoken
enc = tiktoken.get_encoding("o200k_base")

INK, BEIGE, PINK, PINK_D, LIME, GREEN_M = "#3A5615", "#FAF3EC", "#EDA6BF", "#B04874", "#ACC653", "#5E8A1E"
SAR = 3.75   # riyals per US dollar

MESSAGES = ["كم الحد الأعلى للشحن باليوم؟",
            "سحبت فلوس من الصراف وما طلعت لي، بس المبلغ انخصم من حسابي",
            "حولت فلوس لأخوي من أمس وللحين ما وصلت له"]

def msg_cost(model, t_in, t_out):
    p_in, p_out = PRICES[model]
    return (t_in * p_in + t_out * p_out) / 1_000_000

# ---------- one pipeline, running in its own thread ----------
def new_lane(phase="starting"):
    return {"phase": phase, "reply": "", "out": 0, "t0": time.time()}

def run_lane(lane, message, writer):
    try:
        lane["phase"] = "routing"
        r = route("gpt-4o", message)                      # job 1: gpt-4o decides (both designs)
        lane["route"], lane["policy"] = r, POLICY.get(r["pred"], "No policy available.")
        lane["phase"] = "writing"
        system = REPLY_PROMPT.format(policy=lane["policy"])
        lane["in_est"] = len(enc.encode(system + message))
        stream = client.chat.completions.create(          # job 2: the writer, streamed live
            model=writer, temperature=0, max_completion_tokens=120, stream=True,
            stream_options={"include_usage": True},
            messages=[{"role": "system", "content": system}, {"role": "user", "content": message}])
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                lane["reply"] += chunk.choices[0].delta.content
                lane["out"] = len(enc.encode(lane["reply"]))
            if chunk.usage:
                lane["usage"] = (chunk.usage.prompt_tokens, chunk.usage.completion_tokens)
        lane["secs"] = time.time() - lane["t0"]
    except Exception as e:
        lane["error"] = str(e)
    lane["phase"] = "done"

def lane_cost(lane, writer):
    r = lane.get("route")
    if not r:
        return 0.0
    t_in, t_out = lane.get("usage") or (lane.get("in_est", 0), lane.get("out", 0))
    return msg_cost("gpt-4o", r["route_in"], r["route_out"]) + msg_cost(writer, t_in, t_out)

# ---------- drawing ----------
def step(state, title, detail=""):
    return f'<div class="step {state}"><div class="ic"></div><div><div class="st">{title}</div>{detail}</div></div>'

def render_lane(lane, name, sub, writer, color):
    ph, r = lane["phase"], lane.get("route")
    big_chip = f"<span class='chip' style='--c:{PINK_D}'>gpt-4o</span>"
    w_chip = f"<span class='chip'>{writer}</span>"
    if lane.get("error"):
        body = f"<div class='sd' style='color:{PINK_D}'>Error: {html.escape(lane['error'])[:300]}</div>"
    else:
        if r is None:
            s1 = step("wait" if ph == "idle" else "active",
                      f"{big_chip} picks the department" if ph == "idle" else f"{big_chip} reading the message…")
            s2 = step("wait", "Open the matching policy")
        else:
            s1 = step("done", f"{big_chip} decided <b>{html.escape(r['pred'])}</b>",
                      f"<div class='sd'>{r['confidence']:.1%} confident · {r['route_out']} tokens out</div>")
            s2 = step("done", f"Read the <b>{html.escape(r['pred'])}</b> policy",
                      f"<div class='pol'>{html.escape(lane['policy'])}</div>")
        if ph == "writing":
            s3 = step("active", f"{w_chip} writing the Arabic reply…",
                      f"<div class='reply'>{html.escape(lane['reply'])}<span class='caret'></span></div>")
        elif ph == "done":
            s3 = step("done", f"{w_chip} wrote the reply", f"<div class='reply'>{html.escape(lane['reply'])}</div>")
        else:
            s3 = step("wait", f"{w_chip} writes the Arabic reply")
        body = s1 + s2 + s3
    secs = lane.get("secs") or (0 if ph == "idle" else time.time() - lane["t0"])
    meter = (f"<div class='meter'><div><b>{lane['out']}</b>reply tokens</div>"
             f"<div><b>SAR {lane_cost(lane, writer) * 1000 * SAR:.2f}</b>per 1,000 messages</div>"
             f"<div><b>{secs:.1f}s</b>time</div></div>")
    return (f"<div class='lane' style='--c:{color}'><div class='lane-t'>{name}</div>"
            f"<div class='lane-s'>{sub}</div>{body}{meter}</div>")

def render_cmp(b, o):
    cb, co = lane_cost(b, "gpt-4o"), lane_cost(o, "gpt-4o-mini")
    same = b["route"]["pred"] == o["route"]["pred"]
    bar = lambda label, c, w, col: (f"<div class='bar'><span>{label}</span><div class='tr'>"
                                    f"<div style='width:{w:.0f}%;background:{col}'></div></div><b>SAR {c * 1000 * SAR:.2f}</b></div>")
    return (f"<div class='cmp'><div class='cmp-big'>{1 - co / cb:.0%} cheaper</div>"
            f"<div class='cmp-sub'>{'Same decision' if same else 'Different decisions'} · "
            f"{o['secs']:.1f}s vs {b['secs']:.1f}s · cost per 1,000 messages like this one</div>"
            + bar("Baseline", cb, 100, PINK) + bar("My design", co, co / cb * 100, LIME) + "</div>")

def render(base, mine):
    out = ("<div class='lanes'>"
           + render_lane(base, "Baseline", "gpt-4o decides and writes", "gpt-4o", PINK_D)
           + render_lane(mine, "My design", "gpt-4o decides · mini writes", "gpt-4o-mini", GREEN_M) + "</div>")
    if base["phase"] == mine["phase"] == "done" and not (base.get("error") or mine.get("error")):
        out += render_cmp(base, mine)
    return out

# ---------- run both at the same time ----------
def compare(message):
    base, mine = new_lane(), new_lane()
    threading.Thread(target=run_lane, args=(base, message, "gpt-4o"), daemon=True).start()
    threading.Thread(target=run_lane, args=(mine, message, "gpt-4o-mini"), daemon=True).start()
    while not (base["phase"] == "done" and mine["phase"] == "done"):
        yield render(base, mine)
        time.sleep(0.08)
    yield render(base, mine)

CSS = """
body, gradio-app{background:
  radial-gradient(45% 35% at 88% 12%, rgba(58,86,21,.55), transparent 70%),
  radial-gradient(55% 30% at 30% 42%, rgba(172,198,83,.50), transparent 70%),
  radial-gradient(40% 35% at 12% 88%, rgba(237,166,191,.45), transparent 70%),
  radial-gradient(40% 30% at 82% 82%, rgba(172,198,83,.40), transparent 70%),
  #FAF3EC !important; background-attachment:fixed !important}
body::before{content:"";position:fixed;inset:0;pointer-events:none;z-index:0;opacity:.20;mix-blend-mode:multiply;
  background-image:url("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='160' height='160'><filter id='n'><feTurbulence type='fractalNoise' baseFrequency='.9' numOctaves='2'/></filter><rect width='100%' height='100%' filter='url(%23n)' opacity='.6'/></svg>")}
.gradio-container{max-width:1100px !important;margin:0 auto !important;background:transparent !important;position:relative;z-index:1;color:#3A5615}

/* start page */
#start{min-height:80vh;justify-content:center !important;align-items:center !important;gap:34px !important}
.start-title{font-family:'Instrument Serif',Georgia,serif;font-size:80px;line-height:1;text-align:center;color:#3A5615}
#start-btn{flex:none !important;width:180px !important;max-width:180px !important;height:54px !important;border-radius:999px !important;
  background:#3A5615 !important;color:#FAF3EC !important;border:none !important;font-size:18px !important;font-weight:700}
#start-btn:hover{background:#B04874 !important}

/* slim control bar */
#bar{align-items:center !important;gap:12px !important;background:rgba(250,243,236,.7);backdrop-filter:blur(14px);
  border:1px solid rgba(58,86,21,.10);border-radius:22px;padding:6px 6px 6px 10px !important;margin:4px 0 14px}
#pick{background:transparent !important;border:none !important;box-shadow:none !important;padding:0 !important}
#pick .wrap{display:flex !important;flex-wrap:wrap;gap:6px !important}
#pick label{direction:rtl;font-size:15px !important;font-weight:600 !important;padding:7px 14px !important;border-radius:999px !important;
  border:1px solid rgba(58,86,21,.15) !important;background:rgba(255,253,250,.9) !important;color:#3A5615 !important;
  font-family:'IBM Plex Sans Arabic',Tahoma,sans-serif !important;box-shadow:none !important}
#pick label span{font-size:15px !important}
#pick label input{display:none !important}
#pick label.selected{background:#EDA6BF !important;border-color:#EDA6BF !important}
#pick label.selected, #pick label.selected span{color:#3A5615 !important}
#go{flex:none !important;min-width:130px !important;height:42px !important;border-radius:999px !important;
  background:#3A5615 !important;color:#FAF3EC !important;border:none !important;font-weight:700;font-size:15px !important}
#go:hover{background:#B04874 !important}

/* the two cards */
.lanes{display:grid;grid-template-columns:1fr 1fr;gap:16px}
@media(max-width:800px){.lanes{grid-template-columns:1fr}}
.lane{background:rgba(255,253,250,.82);backdrop-filter:blur(14px);color:#3A5615;border:1px solid rgba(58,86,21,.10);border-radius:22px;padding:22px 24px}
.lane-t{font-family:'Instrument Serif',Georgia,serif;font-size:32px;line-height:1.1;color:#3A5615}
.lane-s{font-size:13px;color:#6B7A5E;margin-bottom:8px}
.step{display:grid;grid-template-columns:20px 1fr;gap:10px;padding:12px 0;border-top:1px solid rgba(58,86,21,.08)}
.ic{width:12px;height:12px;border-radius:50%;margin-top:5px;border:2px solid #E3D9CC}
.step.done .ic{background:var(--c);border-color:var(--c)}
.step.active .ic{border-color:var(--c);animation:pulse 1s ease-in-out infinite}
.step.wait .st{color:#B5AC9C}
@keyframes pulse{50%{box-shadow:0 0 0 6px rgba(237,166,191,.35)}}
.st{font-size:15px;color:#3A5615}
.sd{font-size:13px;color:#6B7A5E;margin-top:2px}
.chip{display:inline-block;font-size:12px;font-weight:700;padding:1px 9px;border-radius:99px;color:#fff !important;background:var(--c);margin-right:4px}
.pol{font-size:12.5px;color:#5A4A50;background:rgba(237,166,191,.20);border-radius:10px;padding:8px 10px;margin-top:6px}
.reply{direction:rtl;font-family:'IBM Plex Sans Arabic',Tahoma,sans-serif;font-size:20px;line-height:1.8;margin-top:6px;min-height:36px;color:#3A5615}
.caret{display:inline-block;width:2px;height:1.1em;background:var(--c);vertical-align:middle;margin-right:2px;animation:blink .8s steps(1) infinite}
@keyframes blink{50%{opacity:0}}
.meter{display:flex;gap:24px;margin-top:6px;padding-top:12px;border-top:1px solid rgba(58,86,21,.08);font-variant-numeric:tabular-nums}
.meter div{font-size:12px;color:#6B7A5E}
.meter b{display:block;font-size:24px;color:#3A5615}
.cmp{margin-top:16px;background:#3A5615;border-radius:22px;padding:24px 26px}
.cmp, .cmp *{color:#FAF3EC !important}
.cmp-big{font-family:'Instrument Serif',Georgia,serif;font-size:48px;line-height:1}
.cmp-sub{font-size:14px;opacity:.8;margin:6px 0 14px}
.bar{display:grid;grid-template-columns:100px 1fr 90px;gap:12px;align-items:center;margin-top:10px;font-size:14px}
.tr{height:10px;background:rgba(250,243,236,.16);border-radius:99px;overflow:hidden}
.tr div{height:100%;border-radius:99px}
.bar b{text-align:right;font-variant-numeric:tabular-nums}
"""

theme = gr.themes.Soft(primary_hue="green", neutral_hue="stone",
                       font=[gr.themes.GoogleFont("IBM Plex Sans"), gr.themes.GoogleFont("IBM Plex Sans Arabic"),
                             gr.themes.GoogleFont("Instrument Serif"), "sans-serif"])

with gr.Blocks(theme=theme, css=CSS, title="Arabic Token Tax Router") as app:
    gr.HTML(f"<div style='padding:14px 2px 4px;color:{INK}'>"
            "<div style=\"font-family:'Instrument Serif',Georgia,serif;font-size:42px;line-height:1\">Cutting the Arabic Token Tax</div>"
            "<div style='font-size:15px;opacity:.7;margin-top:6px'>Same customer message, two pipelines, running live side by side</div></div>")
    with gr.Row(equal_height=False, elem_id="bar"):
        msg = gr.Radio(MESSAGES, value=MESSAGES[0], show_label=False, container=False, scale=5, elem_id="pick")
        btn = gr.Button("Compare", variant="primary", scale=0, elem_id="go")
    out = gr.HTML(render(new_lane("idle"), new_lane("idle")))
    btn.click(compare, msg, out, show_progress="hidden")

app.launch(show_error=True)

/tmp/ipykernel_3243/1646695889.py:183: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CSS, title="Arabic Token Tax Router") as app:
/tmp/ipykernel_3243/1646695889.py:183: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CSS, title="Arabic Token Tax Router") as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3a7ffe7fb6c5708f59.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
   !unzip -o -q results.zip
   !ls *.csv